In [ ]:
# Esame 654AA - a.a. 2025/2026
# Studenti: Leonardo Celati, Samuele Taviano
# Matricole: 660185,

In [ ]:
import importlib
from sklearn.model_selection import StratifiedKFold
import wine_common as wine
import knn_common as kc
from sklearn.metrics import ConfusionMatrixDisplay, classification_report



In [ ]:
importlib.reload(wine)
importlib.reload(kc)

In [ ]:
# Split for KFold
n_split = 5
default_cv = StratifiedKFold(n_splits=n_split, shuffle=True, random_state=42)

default_krange = list(range(1, 41, 2))

default_random_state = 42

# Subset is derived from data analysis
# and feature selection, more for learning purpose
# than for real needs as the numner of features is small
subset_features_1 = [
    'alcohol',
    'volatile acidity',
    'sulphates',
    'citric acid',
    'total sulfur dioxide'
]


<h2>Wine Dataset</h2>
<hr/>

<h4>Data Loading</h4>
<p>Load and introspect data from monk training and test set.</p>

In [ ]:
importlib.reload(wine)
df_train, df_test = wine.stratified_split(type='white', seed=default_random_state)
X_tr_full, y_tr, X_ts, y_ts = wine.split_and_prepare_dataset(df_train, df_test)
X_tr = X_tr_full[subset_features_1]
X_ts = X_ts[subset_features_1]
wine.dataset_introspection(df_train, df_test)

In [ ]:
weights = "uniform"
metric = "euclidean"
model, cv_scores, train_scores = kc.find_best_knn_model(
    X_tr, y_tr, default_cv, default_krange,weights=weights,metric=metric
)

best_k, best_mean, best_std = kc.knn_introspection(cv_scores)


In [ ]:
kc.plot_knn_validation_curve(cv_scores, train_scores)

In [ ]:
kc.plot_knn_learning_curve(model, X_tr, y_tr, default_cv)

In [ ]:
importlib.reload(kc)
models=[]
for k in kc.k_neighborhood(best_k, len(y_tr)):
    models.append(KNeighborsClassifier(
        n_neighbors=k,
        weights=weights,
        metric=metric
    ))

kc.plot_knn_learning_curves_grid(
    models=models,
    X=X_tr,
    y=y_tr,
    cv=default_cv,
    highlight_k=best_k
)

<h4>Prediction</h4>
<p>Performing a prediction on the test set</p>

In [ ]:
model.fit(X_tr, y_tr)
y_pred = model.predict(X_ts)

In [ ]:
print(classification_report(y_ts, y_pred, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_ts, y_pred, xticks_rotation='vertical', cmap='Blues');